# *TikTok*-Daten

Dieses Notebook hat zwei Teile. Im ersten Teil schauen wir uns an, wie wir *TikTok*-Daten importieren können (um sie dann zu explorieren oder weiter zu bearbeiten), die mit dem Tool [zeeschuimer](https://github.com/digitalmethodsinitiative/zeeschuimer) gesammelt wurden. Diese Beispiele orientieren sich am [Demo-Notebook für *Instagram*-Daten](https://github.com/jobreu/insta-explore). Im zweiten Teil nutzen wir das `R`-Paket [traktok](https://jbgruber.github.io/traktok/) um Daten von *TikTok* zu sammeln. Hierfür nutzen wir Accounts aus dem Datensatz [*Social Media Accounts (TikTok, YouTube, X/Twitter) of the Candidates in the 2025 German Federal Election*](https://search.gesis.org/research_data/SDN-10.7802-2862), den wir uns auch bereits in der dritten Sitzung des Seminars (am 05.11.2025) angeschaut hatten (das `R`-Skript zur Erstellung dieser Beispieldaten heißt `tiktok_handles.R` und ist ebenfalls im *GitHub* Repository mit diesem Notebook verfügbar).

## Daten aus `zeeschuimer` einlesen

Die einfachste Variante ist, eine `.csv`-Datei einzulesen, die mit [4CAT](https://4cat.nl/) aus einer mit `zeeschuimer` generierten `.ndjson`-Datei erstellt wurde. **NB**: Um die Datei in diesem Notebook importieren zu können, muss diese zunächst in die Online-Umgebung hochgeladen werden. Dies geht über das entsprechende Symbol oben im File Explorer (Upload Files) auf der linken Seite des *Jupyter Lab*-Interface.

In [ ]:
library(readr)

In [ ]:
tiktok1 <- read_csv("INSERT_FILE_NAME_HERE") # Namen der entsprechenden Datei (inkl. Dateiendung) einfügen

In [ ]:
names(tiktok1)

In [ ]:
library(dplyr)

In [ ]:
glimpse(tiktok1)

Als zweite Option können wir einen Parser nutzen, um Daten direkt aus einer mit `zeeschuimer` erstellten `.ndjson`-Datei einzulesen. Der Parser wurde mit "freundlicher Unterstützung" von [GitHub Copilot](https://github.com/features/copilot) geschrieben. Auch hierfür muss die entsprechende Datei wie im Beispiel mit dem `.csv` File hochgeladen werden.

In [ ]:
source("parse_tiktok.R")

In [ ]:
tiktok2 <- parse_4cat_ndjson_to_tibble("INSERT_FILE_NAME_HERE") # Namen der entsprechenden Datei (inkl. Dateiendung) einfügen

In [ ]:
names(tiktok2)

In [ ]:
glimpse(tiktok2)

## `traktok`

Nachfolgend schauen wir uns an, wie wir *TikTok*-Daten mit dem `R`-Paket [traktok](https://github.com/JBGruber/traktok/) sammeln können. Die Beispiele orientieren sich an der [Dokumentationsseite für das Paket](https://jbgruber.github.io/traktok/index.html).

**NB**: Die nachfolgenden Code-Beispiele mit `traktok` nutzen die "hidden API" von *TikTok* und einen Web-Scraping-Ansatz. Es kann sein, dass nicht alle Funktionen direkt/einwandfrei funktionieren; insbesondere in einem *Jupyter Notebook* und über *Binder*. Es lohnt sich daher ggf., nach Möglichkeit den `R`-Code lokal auszuführen.

`traktok` bietet auch Funktionen zur Nutzung der offiziellen [*TikTok* Research API](https://developers.tiktok.com/products/research-api/). Für diese muss man sich allerdings bewerben.

### Setup

Für die Nutzung der "hidden API" ist es nötig, Cookies zu speichern (siehe https://jbgruber.github.io/traktok/articles/unofficial-api.html). Dies geht am einfachsten mit den Browser-Erweiterungen [Get cookies.txt](https://chromewebstore.google.com/detail/get-cookiestxt-locally/cclelndahbckbenkjhflpdbgdldlbecc) für *Chrome* bzw. [cookies.txt](https://addons.mozilla.org/en-US/firefox/addon/cookies-txt/) für *Firefox*. 

Sobald die `.txt`-Datei mit den Cookies für *TikTok* gespeichert wurde, kann diese mit einer Funktion aus dem Paket [`cookiemonster`](https://github.com/JBGruber/cookiemonster) zur Authentifizierung genutzt werden. Auch hierfür muss in der Online-Umgebung die Datei hochgeladen werden (s.o.).

In [ ]:
library(cookiemonster)
library(traktok)

In [ ]:
add_cookies("cookies.txt")

Für die meisten Funktionen von `traktok` braucht man die User- oder Video-IDs. Die User ID können wir für einen bestimmten User Name wie folgt finden.

### Daten sammeln

Die Accounts in den folgenden Beispielen stammen aus dem Datensatz [*Social Media Accounts (TikTok, YouTube, X/Twitter) of the Candidates in the 2025 German Federal Election*](https://search.gesis.org/research_data/SDN-10.7802-2862).

In [ ]:
user_info <- tt_user_info_hidden("franziska.brantner")
user_info

Mit der User-ID können wir z.B. Informationen zu den Accounts sammeln, denen dieser User folgt.

In [ ]:
following <- tt_get_following_hidden(secuid = user_info$secUid,
                                     verbose = TRUE)

In [ ]:
following

Wir können auch Informationen über die Videos eines bestimmten Accounts sammeln. Mit der Funktion, die wir dazu nutzen, lassen sich die Videos auch herunterladen. Das wollen wir hier aber nicht machen.

In [ ]:
videos <- tt_user_videos_hidden("franziska.brantner", 
                                save_video = FALSE)

**Wichtiger Hinweis**: Da `traktok` im Hintergrund Browsing-Verhalten simuliert kann es sein, dass Checks für automasierte Nutzung, wie z.B. das Lösen von CAPTCHAs nötig sind. Um dies machen zu können, muss der entsprechende `R`-Code interaktiv ausgeführt werden (d.h. am besten lokal über *RStudio*). Hierfür muss dann im Befel `tt_user_videos_hidden()` das zusätzliche Argument `solve_captchas = TRUE` setzen.

In [ ]:
videos

Analog zu den [Beispielen für *Blueksy* mit dem `R`-Paket `atrrr`](https://github.com/jobreu/atrrr-demo) können wir natürlich auch mit einem kombinierten Befehl Daten für mehrere Accounts sammeln.

In [ ]:
library(purrr)

In [ ]:
accounts <- c("franziska.brantner", "judith.skudelny", "jamila.schaefer", "martin.hagen")

In [ ]:
users <- map_df(
  accounts,
  ~ tt_user_info_hidden(.x)
)

In [ ]:
users